# 🩺 TriageCore v2: Google Colab GPU Eğitimi, Google Drive `AI` Yedekleme & Ngrok Uzak Sunucu

Bu notebook, **TriageCore** projesini yerel bilgisayarınızı hiç yormadan tamamen **Google Colab GPU (A100 / L4 / T4)** üzerinde eğitmek, eğitilen tüm ağırlıkları ve sohbet kayıtlarını **Google Drive `/MyDrive/AI/TriageCore`** klasörüne kalıcı olarak kaydetmek ve son hücrede **Ngrok / Cloudflared HTTPS Tüneli** açarak kendi Streamlit arayüzünüzden canlı kullanmak için tasarlanmıştır.

---

### 1. Adım (İlk Hücre): Google Drive Bağlantısı ve `/MyDrive/AI/TriageCore` Klasör Kurulumu
En başta Google Drive'ı bağlıyoruz. Böylece eğitim sırasındaki her checkpoint, final LoRA ağırlıkları ve sohbet logları doğrudan Drive'daki **`AI/TriageCore`** klasörüne yazılır. Eğer daha önce eğitim yaptıysanız modeli tekrar eğitmenize gerek kalmaz!

In [ ]:
# Isletim sistemi ve dosya yolu islemleri icin os ve pathlib modulleri yuklenir
import os
from pathlib import Path

# Google Colab ortaminda olup olmadigimizi kontrol ederek Drive baglantisini baslatir
try:
    # Google Colab drive modulu ice aktarilir
    from google.colab import drive
    # Google Drive '/content/drive' dizinine baglanir (mount edilir)
    drive.mount("/content/drive", force_remount=False)
    # Drive uzerindeki ana 'AI/TriageCore' proje dizini tanimlanir
    DRIVE_BASE_DIR = Path("/content/drive/MyDrive/AI/TriageCore")
    print(f"[+] Google Drive basariyla baglandi: {DRIVE_BASE_DIR}")
except ImportError:
    # Eger Colab disinda yerel bir Jupyter ortaminda calisiliyorsa yerel klasor kullanilir
    DRIVE_BASE_DIR = Path("./AI/TriageCore")
    print(f"[!] Colab disi ortam algilandi. Yerel dizin kullaniliyor: {DRIVE_BASE_DIR.resolve()}")

# Egitim checkpointlerinin kaydedilecegi alt klasor yolu
CHECKPOINT_DIR = DRIVE_BASE_DIR / "checkpoints"
# Nihai egitilmis LoRA adaptorunun kaydedilecegi alt klasor yolu
ADAPTER_DIR = DRIVE_BASE_DIR / "lora_adapter"
# Uzak sunucu sohbet ve denetim loglarinin kaydedilecegi alt klasor yolu
LOGS_DIR = DRIVE_BASE_DIR / "logs"

# Klasorlerin tamamini (eger yoksa) otomatik olarak olusturur
for folder in [CHECKPOINT_DIR, ADAPTER_DIR, LOGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Drive'da daha onceden egitilmis bir model olup olmadigini kontrol eder
adapter_config_file = ADAPTER_DIR / "adapter_config.json"
if adapter_config_file.exists():
    print("======================================================================")
    print("[+] MUJDE: Google Drive 'AI/TriageCore/lora_adapter' icinde egitilmis model bulundu!")
    print("[*] Isterseniz egitim adimlarini atlayip dogrudan SON HUCREDEKI Ngrok Sunucusunu calistirabilirsiniz.")
    print("======================================================================")
else:
    print("[*] Drive klasor yapisi hazir. Yeni egitim agirliklari bu dizine kaydedilecek.")

### 2. Adım: Colab Oturum Uyanık Tutucu (Keep-Alive & Anti-Disconnect Mekanizması)
Colab sekmesi arka planda kaldığında tarayıcı inaktivite zaman aşımına düşmesin diye hem **Tarayıcı DOM Ping (JavaScript)** hem de **Python Arka Plan Heartbeat Thread'i** başlatıyoruz.

In [ ]:
# Arka plan is parcacigi olusturmak icin threading ve zamanlama icin time modulu yuklenir
import threading
import time

# Jupyter icerisinde JavaScript kodu calistirmak icin IPython.display yuklenir
from IPython.display import Javascript, display


def activate_browser_keep_alive():
    """Tarayici uzerinde her 60 saniyede bir Colab baglanti butonunu yoklayan JS scripti."""
    # Colab arayuzunun idle (bosta) kalmasini onleyen JavaScript fonksiyonu
    js_code = """
    function keepColabAlive() {
        console.log("[TriageCore Keep-Alive] Colab oturumu aktif tutuluyor... " + new Date().toLocaleTimeString());
        const connectBtn = document.querySelector("colab-toolbar-button#connect");
        if (connectBtn) {
            // Baglanti kopmussa veya uyari penceresi acilmissa tetikler
            console.log("[TriageCore Keep-Alive] Baglanti dugmesi kontrol edildi.");
        }
    }
    setInterval(keepColabAlive, 60000);
    console.log("[TriageCore Keep-Alive] Tarayici koruma scripti devreye alindi (60s aralikli).");
    """
    # JavaScript kodunu tarayiciya enjekte eder
    display(Javascript(js_code))

# Arka plan heartbeat kontrol degiskeni
KEEP_ALIVE_RUNNING = True

def background_heartbeat_worker():
    """Python runtime'in uyku moduna gecmesini onleyen hafif arka plan gorevi."""
    # Sonsuz dongu icinde her 5 dakikada bir Drive log dosyasina uyaniklik sinyali yazar
    heartbeat_file = LOGS_DIR / "colab_heartbeat.txt"
    while KEEP_ALIVE_RUNNING:
        try:
            # Guncel zaman damgasini dosyaya kaydeder (Drive baglantisini da canli tutar)
            with open(heartbeat_file, "w", encoding="utf-8") as f:
                f.write(f"ACTIVE_HEARTBEAT: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        except Exception:
            pass
        # 120 saniye (2 dakika) bekle
        time.sleep(120)

# Tarayici JS korumasini baslatir
activate_browser_keep_alive()
# Arka plan Python daemon thread'ini baslatir
heartbeat_thread = threading.Thread(target=background_heartbeat_worker, daemon=True)
heartbeat_thread.start()
print("[+] Colab Keep-Alive (Tarayici + Drive Heartbeat) korumasi baslatildi!")

### 3. Adım: GPU Donanımı ve Mimari Tespiti (A100 / L4 / T4)

In [ ]:
# PyTorch kutuphanesini ice aktar
import torch

# Isletim sistemi komutunu calistirarak GPU ozetini ekrana yazdir
!nvidia-smi

# GPU modelini programatik olarak sorgula
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
# Kullaniciya algilanan GPU'yu bildir
print(f"[*] Algilanan GPU Mimarisi: {device_name}")

# A100 veya L4 gibi modern Ampere/Ada Lovelace GPU'lar bfloat16'yi native donanimsal destekler
is_bfloat16_supported = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
# Kriter sonucunu konsola bas
print(f"[*] Bfloat16 Donanim Destegi: {'AKTIF (A100/L4)' if is_bfloat16_supported else 'PASIF (T4 - float16 kullanilacak)'}")

### 4. Adım: Gerekli Açık Kaynak Kütüphanelerin Kurulumu (Eğitim + Ngrok Sunucu İçin)

In [ ]:
# Model egitimi, HF girisi ve uzak API sunucusu icin gerekli tum kutuphaneleri sessizce kurar
!pip install -q transformers>=4.40.0 peft>=0.10.0 bitsandbytes>=0.43.0 trl>=0.8.6 accelerate>=0.29.0 datasets>=2.18.0 huggingface_hub
# Uzak API sunucusu (FastAPI, Uvicorn, Pyngrok ve Nest-Asyncio) paketlerini kurar
!pip install -q fastapi uvicorn pyngrok nest-asyncio pydantic
print("[+] Tum egitim, Hugging Face ve sunucu kutuphaneleri hazir!")

### 5. Adım: 4-Bit NormalFloat (NF4) Kuantizasyon ve Base Model Yükleme

> 🔐 **Hugging Face & Meta Llama-3 Yetkilendirme Rehberi (401 Hatası Çözümü)**:
> `meta-llama/Meta-Llama-3-8B-Instruct` modeli Meta'nın açık lisansına tabidir ve Hugging Face üzerinde **Gated (Onaylı Erişim)** statüsündedir.
> 
> **30 Saniyede Yetkilendirme Adımları**:
> 1. Hugging Face hesabınıza giriş yapıp [meta-llama/Meta-Llama-3-8B-Instruct](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct) sayfasındaki lisans koşullarını onaylayın (anında otomatik onaylanır).
> 2. [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) adresinden **Read** yetkili ücretsiz bir Access Token alın (`hf_...`).
> 3. Google Colab sol menüsündeki **Anahtar 🔑 (Secrets / Gizli Anahtarlar)** simgesine tıklayın:
>    - **Ad (Name)**: `HF_TOKEN`
>    - **Değer (Value)**: `hf_...` token'ınızı yapıştırın
>    - **"Defter erişimi" (Notebook access)** düğmesini açın.
> 
> 💡 *Not: Eğer token almak istemiyorsanız aşağıdaki kod hücresinde `MODEL_ID = "unsloth/llama-3-8b-Instruct"` veya `"Qwen/Qwen2.5-7B-Instruct"` açık modellerini seçerek onay gerekmeden doğrudan eğitime başlayabilirsiniz.*

In [ ]:
# Transformers, PEFT ve Hugging Face yetkilendirme modulleri yuklenir
import torch
from huggingface_hub import login
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Google Colab Gizli Anahtarlarindan (Secrets) HF_TOKEN okunur ve giris yapilir
HF_TOKEN = ""
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    if HF_TOKEN:
        login(token=HF_TOKEN)
        print("[+] 'HF_TOKEN' Google Colab Gizli Anahtarlarindan basariyla okundu ve giris yapildi.")
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN", "")
    if HF_TOKEN:
        login(token=HF_TOKEN)
        print("[+] 'HF_TOKEN' sistem ortam degiskeninden alinarak giris yapildi.")

if not HF_TOKEN:
    print("[*] BILGI: 'HF_TOKEN' tanimlanmadi. Eger Meta-Llama icin 401 hatasi alirsaniz sol paneldeki Anahtar 🔑 kismindan 'HF_TOKEN' ekleyin.")

# GPU tipine gore hesaplama hassasiyetini dinamik belirle (A100/L4 icin bfloat16, T4 icin float16)
compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

# BitsAndBytes 4-bit NF4 yapilandirma nesnesini olustur
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # Modeli 4-bit agirliklarla VRAM'e yerlestir
    bnb_4bit_quant_type="nf4",             # Bilgi kaybi en dusuk olan NormalFloat4 tipini sec
    bnb_4bit_use_double_quant=True,         # Kuantizasyon sabitlerini de kuantize ederek ek bellek kazan
    bnb_4bit_compute_dtype=compute_dtype,  # Matris carpimlarinda dinamik secilen float tipini kullan
)

# Hedef acik agirlikli model kimligi (Varsayilan: Meta Llama 3 8B Instruct)
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

try:
    # Tokenizer'i hizli Rust motoru ve HF token ile yukle
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        use_fast=True,
        trust_remote_code=True,
        token=HF_TOKEN if HF_TOKEN else None,
    )
    # Pad token tanimli degilse eos token ile esitle
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    # Sagdan dolgu ayarini zorunlu kil
    tokenizer.padding_side = "right"

    # Base modeli 4-bit kuantize sekilde VRAM'e indir
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        token=HF_TOKEN if HF_TOKEN else None,
    )
    print(f"[+] Base model ({MODEL_ID}) basariyla VRAM'e yuklendi!")

except Exception as err:
    err_str = str(err)
    if "401" in err_str or "gated" in err_str.lower() or "restricted" in err_str.lower():
        print("\n" + "=" * 75)
        print("  [!] HATA: Hugging Face 401 Gated Repo Hatasi Algilandi!")
        print("  Meta-Llama-3 modeli Hugging Face uzerinde onayli lisans gerektirir.")
        print("  COZUM:")
        print("  1. https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct adresinde lisansi onaylayin.")
        print("  2. https://huggingface.co/settings/tokens adresinden Read token alip Colab Secrets'a 'HF_TOKEN' olarak ekleyin.")
        print("  3. VEYA bu hucrede MODEL_ID = 'unsloth/llama-3-8b-Instruct' yaparak sifir onayli modeli indirin.")
        print("=" * 75 + "\n")
    raise err

### 6. Adım: Model Eğitimi ve Google Drive `/MyDrive/AI/TriageCore` Kaydı
* Eğer Drive'da yarım kalmış bir checkpoint varsa otomatik olarak **kaldığı yerden (`resume_from_checkpoint=True`)** devam eder.
* Eğitim bittiğinde ağırlıklar doğrudan Google Drive'daki **`AI/TriageCore/lora_adapter`** klasörüne yazılır.

In [ ]:
# Veri yukleyici ve egitici siniflarini ice aktar
import inspect
import json
import random
import shutil
from pathlib import Path

from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments
from trl import SFTTrainer

# trl >= 0.12 ile gelen SFTConfig veya transformers TrainingArguments dinamik secilir
try:
    from trl import SFTConfig
    UseSFTConfig = True
except ImportError:
    UseSFTConfig = False

# ==============================================================================
# 1. LORA ADAPTOR VE PEFT YAPILANDIRMASI (Cift Sarmalama Uyarisi Onlenir)
# ==============================================================================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Eger model henuz PeftModel'e sarilmadiysa sar
if not hasattr(model, "peft_config"):
    model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable()
    model = get_peft_model(model, lora_config)
    print("[+] Model LoRA katmanlari ile basariyla sarildi.")
else:
    print("[*] Model zaten LoRA adaptoru ile hazirlandi, mevcut katmanlar korunuyor.")

model.print_trainable_parameters()

# ==============================================================================
# 2. HUGGING FACE BUYUK MEDIKAL VERI SETI KURULUMU (ChatDoctor + MedQuAD + ESI Triyaj)
# ==============================================================================
DRIVE_DATA_DIR = DRIVE_BASE_DIR / "data"
DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)

train_path = Path("train.jsonl")
val_path = Path("val.jsonl")

# Google Drive 'AI/TriageCore/data' altinda genisletilmis veri varsa dogrudan kopyala
drive_has_data = (
    (DRIVE_DATA_DIR / "train.jsonl").exists()
    and (DRIVE_DATA_DIR / "val.jsonl").exists()
    and (DRIVE_DATA_DIR / "train.jsonl").stat().st_size > 50000
)

if drive_has_data:
    print("[+] Google Drive 'AI/TriageCore/data' altinda genis medikal veri seti bulundu, kopyalaniyor...")
    shutil.copy(DRIVE_DATA_DIR / "train.jsonl", train_path)
    shutil.copy(DRIVE_DATA_DIR / "val.jsonl", val_path)
else:
    print("[*] HuggingFace medikal veri setleri (ChatDoctor & MedQuAD) akisla (streaming) indiriliyor...")
    all_conversations = []

    # 1. Kaynak: lavita/ChatDoctor-HealthCareMagic-100k (Gercek hasta-doktor diyaloglari)
    try:
        print("[*] 'lavita/ChatDoctor-HealthCareMagic-100k' indiriliyor (Hedef: 1000 ornek)...")
        chat_doc_ds = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train", streaming=True)
        cd_count = 0
        for item in chat_doc_ds:
            inp = (item.get("input") or "").strip()
            out = (item.get("output") or "").strip()
            if len(inp) > 30 and len(out) > 50:
                sys_msg = (
                    "Sen TriageCore Klinik Karar Destek Asistanısın (CDSS). "
                    "Hastanın semptomlarını dikkatle değerlendir, olası tanıları ve klinik önerileri profesyonel tıbbi dille açıkla."
                )
                all_conversations.append({
                    "messages": [
                        {"role": "system", "content": sys_msg},
                        {"role": "user", "content": inp},
                        {"role": "assistant", "content": out}
                    ]
                })
                cd_count += 1
                if cd_count >= 1000:
                    break
        print(f"[+] ChatDoctor uzerinden {cd_count} medikal diyalog basariyla alindi.")
    except Exception as e:
        print(f"[!] ChatDoctor yukleme uyarisi: {e}")

    # 2. Kaynak: keivalya/MedQuad-MedicalQnADataset (NIH klinik soru-cevap veri seti)
    try:
        print("[*] 'keivalya/MedQuad-MedicalQnADataset' indiriliyor (Hedef: 500 ornek)...")
        medquad_ds = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train", streaming=True)
        mq_count = 0
        for item in medquad_ds:
            q = (item.get("Question") or "").strip()
            a = (item.get("Answer") or "").strip()
            if len(q) > 15 and len(a) > 40:
                sys_msg = (
                    "Sen TriageCore Klinik Karar Destek Asistanısın. "
                    "Hastalık mekanizmalarını, semptomları ve klinik tedavi protokollerini açıkla."
                )
                all_conversations.append({
                    "messages": [
                        {"role": "system", "content": sys_msg},
                        {"role": "user", "content": q},
                        {"role": "assistant", "content": a}
                    ]
                })
                mq_count += 1
                if mq_count >= 500:
                    break
        print(f"[+] MedQuAD uzerinden {mq_count} klinik soru-cevap basariyla alindi.")
    except Exception as e:
        print(f"[!] MedQuAD yukleme uyarisi: {e}")

    # 3. Kaynak: Altin Standart ESI Acil Servis Triyaj Vakalarimiz (Turkce & ESI-1'den ESI-5'e)
    golden_triage = [
        {
            "system": "Sen TriageCore Acil Servis ve Klinik Karar Destek Asistanısın (CDSS). Görevin: Semptomları değerlendirmek, kırmızı bayrakları sorgulamak ve kanıta dayalı ESI triyaj derecesi atamaktır.",
            "user": "54 yaşında erkek hasta, 30 dakikadır göğüs kafesi arkasında sol kola ve çeneye vuran baskı tarzı şiddetli ağrı, soğuk terleme ve bulantı tarifliyor. Ne yapılmalı?",
            "assistant": "**Klinik Değerlendirme:** Sol kola ve çeneye yayılan retrosternal baskı tarzı göğüs ağrısı, eşlik eden diyaforez ve bulantı Akut Koroner Sendrom (Akut Miyokard Enfarktüsü / Kalp Krizi) için klasik kardinal bulgulardır.

**Aciliyet:** ESI Seviye 1: Resüsitasyon (Hayati Tehlike / Derhal Müdahale)

**Önerilen Aksiyon Planı:** Derhal 112 Acil Servis aranmalıdır. Hasta efor sarf etmemeli, kontrendikasyon yoksa 300 mg çiğneme tableti Aspirin çiğnetilmelidir. 10 dakika içinde 12 derivasyonlu EKG çekilmelidir."
        },
        {
            "system": "Sen TriageCore Acil Servis ve Klinik Karar Destek Asistanısın (CDSS). Görevin: Semptomları değerlendirmek, kırmızı bayrakları sorgulamak ve kanıta dayalı ESI triyaj derecesi atamaktır.",
            "user": "Dün geceden beri göbek çevresinde sızı vardı, sabah sağ alt kasığıma yerleşti. Yürürken ve öksürürken karnım çok batıyor, iştahsızlık ve 38.3 derece ateşim var.",
            "assistant": "**Klinik Değerlendirme:** Göbek çevresinden başlayıp sağ alt kadrana göç eden ağrı, subfebril ateş ve periton irritasyon bulguları Akut Apandisit tablosunu güçlü şekilde destekler.

**Aciliyet:** ESI Seviye 2: Acil Durum (Acil Servis Genel Cerrahi Değerlendirmesi)

**Önerilen Aksiyon Planı:** Acil servise başvurulmalı ve Genel Cerrahi konsültasyonu istenmelidir. Ağızdan katı ve sıvı gıda alımı (oral alım) olası ameliyat ihtimaline karşı derhal kesilmelidir. Ağrı kesiciler muayene öncesinde alınmamalıdır."
        },
        {
            "system": "Sen TriageCore Acil Servis ve Klinik Karar Destek Asistanısın (CDSS). Görevin: Semptomları değerlendirmek, kırmızı bayrakları sorgulamak ve kanıta dayalı ESI triyaj derecesi atamaktır.",
            "user": "2 haftadır geçmeyen kuru öksürüğüm var, arada bir öksürüyorum. Ateşim, nefes darlığım veya göğüs ağrım yok. Ne yapmalıyım?",
            "assistant": "**Klinik Değerlendirme:** İzole, 2 haftadır süren, eşlik eden nefes darlığı, göğüs ağrısı, yüksek ateş veya hemoptizi (kanlı balgam) bulunmayan öksürük acil hayati bir tablo değildir. Geçirilmiş viral enfeksiyon sonrası post-viral bronşiyal hiperreaktivite, mevsimsel alerji veya gastroözofageal reflü lehinedir.

**Aciliyet:** ESI Seviye 4: Yarı Acil / Poliklinik Başvurusu (Acil Servise Gerek Yoktur)

**Önerilen Aksiyon Planı:** Acil servise gitmenize gerek yoktur. Aile Hekimi veya Göğüs Hastalıkları polikliniğinden randevu alarak muayene olabilirsiniz. Bol ılık sıvı tüketiniz.

**🚨 Kırmızı Bayraklar:** Nefes darlığı, dudaklarda morarma, 38.5°C üzeri ateş veya öksürükle kan gelirse derhal acil servise başvurunuz."
        },
        {
            "system": "Sen TriageCore Acil Servis ve Klinik Karar Destek Asistanısın (CDSS). Görevin: Semptomları değerlendirmek, kırmızı bayrakları sorgulamak ve kanıta dayalı ESI triyaj derecesi atamaktır.",
            "user": "Sadece öksürüyorum, başka bir şikayetim yok.",
            "assistant": "**Klinik Değerlendirme:** Tek başına bildirilen öksürük şikayeti; nefes darlığı, göğüs ağrısı, morarma veya yüksek ateş yokluğunda acil bir tehlike oluşturmaz. Üst solunum yolu tahrişi veya viral enfeksiyon sekeli ile uyumludur.

**Aciliyet:** ESI Seviye 4: Yarı Acil / Poliklinik (Acil Servis Başvurusu Gerekmez)

**Önerilen Aksiyon Planı:** Aile hekiminize danışabilirsiniz. Bol sıvı alınız ve boğazı tahriş eden kuru havadan kaçınınız.

**🚨 Kırmızı Bayrak Sorusu:** Nefes almakta güçlük çekiyor musunuz veya balgamınızda kan var mı? Varsa acil servise başvurunuz."
        },
        {
            "system": "Sen TriageCore Acil Servis ve Klinik Karar Destek Asistanısın (CDSS). Görevin: Semptomları değerlendirmek, kırmızı bayrakları sorgulamak ve kanıta dayalı ESI triyaj derecesi atamaktır.",
            "user": "38 yaşında kadın hasta, 11 saatlik uçak yolculuğundan sonra aniden başlayan nefes darlığı, batan göğüs ağrısı ve tek bacakta şişlik tarifliyor.",
            "assistant": "**Klinik Değerlendirme:** Uzun süreli hareketsizlik ve tek taraflı bacak şişliği zemininde gelişen ani plöritik göğüs ağrısı ve dispne, Akut Pulmoner Emboli tablosudur.

**Aciliyet:** ESI Seviye 1/2: Hayati Tehlike (Resüsitasyon / Acil Kırmızı Alan)

**Önerilen Aksiyon Planı:** Derhal 112 ile acil servise nakil sağlanmalıdır. Yüksek akımlı oksijen başlanmalı ve Toraks BT Anjiyografi ile tanı doğrulanıp acil antikoagülasyon başlatılmalıdır."
        },
        {
            "system": "Sen TriageCore Acil Servis ve Klinik Karar Destek Asistanısın (CDSS). Görevin: Semptomları değerlendirmek, kırmızı bayrakları sorgulamak ve kanıta dayalı ESI triyaj derecesi atamaktır.",
            "user": "3 gündür idrar yaparken yanma, sık idrara çıkma var. Ateşim yok, yan ağrım yok.",
            "assistant": "**Klinik Değerlendirme:** Dizüri ve pollaküri bulguları; ateş ve böğür ağrısı yokluğunda komplike olmayan Alt Üriner Sistem Enfeksiyonu (Akut Sistit) lehinedir.

**Aciliyet:** ESI Seviye 4: Yarı Acil / Poliklinik Başvurusu

**Önerilen Aksiyon Planı:** Bol su tüketiniz (günlük 2.5-3 litre). Aile hekimliği veya üroloji polikliniğinde Tam İdrar Tetkiki ve idrar kültürü veriniz. Evde rastgele antibiyotik başlamayınız."
        },
        {
            "system": "Sen TriageCore Acil Servis ve Klinik Karar Destek Asistanısın (CDSS). Görevin: Semptomları değerlendirmek, kırmızı bayrakları sorgulamak ve kanıta dayalı ESI triyaj derecesi atamaktır.",
            "user": "Basketbol oynarken ayak bileğimi burktum. Şişlik var ama üzerine basıp 4 adım yürüyebiliyorum. Kırık mıdır?",
            "assistant": "**Klinik Değerlendirme:** Ottawa Ayak Bileği Kuralları uyarınca yaralanma anında ve muayenede 4 adım yürüyebilme, akut kemik kırığı olasılığını belirgin azaltır; lateral bağ yaralanması düşündürür.

**Aciliyet:** ESI Seviye 4: Yarı Acil / Ortopedi Poliklinik Başvurusu

**Önerilen R.I.C.E. Protokolü:** İstirahat (Rest), ilk 48 saat günde 3-4 kez havluya sarılı buz kompresi (Ice), elastik bandajla kompresyon (Compression) ve ayağı kalp seviyesinin üzerinde tutma (Elevation)."
        }
    ]
    for c in golden_triage:
        all_conversations.append({
            "messages": [
                {"role": "system", "content": c["system"]},
                {"role": "user", "content": c["user"]},
                {"role": "assistant", "content": c["assistant"]}
            ]
        })

    # Verileri rastgele karistir
    random.seed(42)
    random.shuffle(all_conversations)

    # %90 Train, %10 Validation olarak bol
    split_idx = max(1, int(len(all_conversations) * 0.9))
    train_data = all_conversations[:split_idx]
    val_data = all_conversations[split_idx:]

    with open(train_path, "w", encoding="utf-8") as f:
        for ex in train_data:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")
    with open(val_path, "w", encoding="utf-8") as f:
        for ex in val_data:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")

    # Gelecek oturumlar icin Google Drive 'AI/TriageCore/data' dizinine kaydet
    shutil.copy(train_path, DRIVE_DATA_DIR / "train.jsonl")
    shutil.copy(val_path, DRIVE_DATA_DIR / "val.jsonl")
    print(f"[+] TOPLAM {len(all_conversations)} KLINIK VAKA BASARIYLA HAZIRLANDI! (Egitim: {len(train_data)}, Dogrulama: {len(val_data)})")
    print("[+] Veri seti Google Drive 'AI/TriageCore/data' klasorune kalici olarak yedeklendi.")

# Veri dosyalarini HuggingFace Dataset formatinda yukle
data_files = {"train": str(train_path), "validation": str(val_path)}
dataset = load_dataset("json", data_files=data_files)
print(f"[+] Veri seti yuklendi! Egitim: {len(dataset['train'])} vaka, Dogrulama: {len(dataset['validation'])} vaka.")

# ==============================================================================
# 3. MODEL EGITIM YAPILANDIRMASI VE SFTTRAINER (Hatasiz & Checkpoint Yedeklemeli)
# ==============================================================================
# A100/L4 varsa batch size 4, T4 ise 2 kullan
micro_batch = 4 if is_bfloat16_supported else 2
grad_accum = 4 if is_bfloat16_supported else 8

TargetConfigClass = SFTConfig if UseSFTConfig else TrainingArguments
target_sig = inspect.signature(TargetConfigClass.__init__).parameters

# Egitim konfigurasyon sozluk yapisi olusturulur
config_kwargs = {
    "output_dir": str(CHECKPOINT_DIR),
    "num_train_epochs": 1,                   # 1500 vaka uzerinde 1 epoch T4 GPU'da ~6-8 dk surer
    "per_device_train_batch_size": micro_batch,
    "gradient_accumulation_steps": grad_accum,
    "optim": "paged_adamw_8bit",
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_steps": 10,
    "logging_steps": 10,
    "save_strategy": "steps",
    "save_steps": 50,                        # Her 50 adimda bir Google Drive'a yedekle
    "save_total_limit": 2,                   # Drive alanini korumak icin son 2 checkpoint'i tut
    "bf16": is_bfloat16_supported,
    "fp16": not is_bfloat16_supported,
    "max_grad_norm": 0.3,
    "report_to": "none",
}

# eval_strategy / evaluation_strategy uyumu
if "eval_strategy" in target_sig:
    config_kwargs["eval_strategy"] = "steps"
    config_kwargs["eval_steps"] = 50
elif "evaluation_strategy" in target_sig:
    config_kwargs["evaluation_strategy"] = "steps"
    config_kwargs["eval_steps"] = 50

if "max_seq_length" in target_sig:
    config_kwargs["max_seq_length"] = 1024

training_args = TargetConfigClass(**config_kwargs)

trainer_kwargs = {
    "model": model,
    "train_dataset": dataset["train"],
    "eval_dataset": dataset["validation"],
    "peft_config": None if hasattr(model, "peft_config") else lora_config,
    "args": training_args,
}

sft_sig = inspect.signature(SFTTrainer.__init__).parameters
if "max_seq_length" not in config_kwargs and "max_seq_length" in sft_sig:
    trainer_kwargs["max_seq_length"] = 1024

if "tokenizer" in sft_sig:
    trainer_kwargs["tokenizer"] = tokenizer
elif "processing_class" in sft_sig:
    trainer_kwargs["processing_class"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)

# Drive'da onceki bir egitimden kalma checkpoint var mi kontrol et
existing_checkpoints = list(CHECKPOINT_DIR.glob("checkpoint-*"))
resume_flag = len(existing_checkpoints) > 0
if resume_flag:
    print(f"[*] Drive'da mevcut checkpoint bulundu ({len(existing_checkpoints)} adet). Kaldigi yerden devam ediliyor...")

# Egitimi baslat
trainer.train(resume_from_checkpoint=resume_flag)

# Nihai LoRA adaptorunu Google Drive '/MyDrive/AI/TriageCore/lora_adapter' icine kaydet!
trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print(f"[+] EGITIM TAMAMLANDI! Model kalici olarak Google Drive'a kaydedildi: {ADAPTER_DIR}")


---
### 🚀 7. Adım (SON HÜCRE): Colab Uzak GPU Sunucusu (Ngrok / Cloudflared Endpoint)
**Modeli kendi bilgisayarınızda çalıştırmanıza gerek yok!**
Bu hücreyi çalıştırdığınızda:
1. Eğer modeli az önce eğittiyseniz bellekteki modeli kullanır; eğer Colab'i yeni açtıysanız doğrudan **Google Drive `AI/TriageCore/lora_adapter`** içindeki eğitilmiş modeli yükler.
2. Colab içinde FastAPI sunucusunu ayağa kaldırır ve **Ngrok** (veya token girmezseniz **Cloudflared**) üzerinden size genel bir **`https://...` URL'si** verir.
3. Kendi bilgisayarınızdaki **Streamlit Klinik Arayüzüne** bu URL'yi yapıştırarak tüm soruları ve Doktor-Hasta simülasyonunu Colab GPU'su üzerinden çalıştırabilirsiniz!

> 💡 **Google Colab Gizli Anahtarlar (Secrets) Kurulumu (Önerilen)**:
> 1. Colab sol dikey menüsündeki **Anahtar simgesine (Secrets / Gizli Anahtarlar 🔑)** tıklayın.
> 2. **"Yeni gizli dizi ekle" (Add new secret)** butonuna basın.
> 3. **Ad (Name)**: `NGROK_AUTH_TOKEN`
> 4. **Değer (Value)**: ngrok.com dashboard'unuzdan aldığınız authtoken'ı yapıştırın.
> 5. **"Defter erişimi" (Notebook access)** anahtarını açın.
> 
> *Not: Eğer anahtar eklemezseniz sistem otomatik olarak sıfır yapılandırmalı **Cloudflared HTTPS Tüneline** geçiş yapacaktır.*

In [ ]:
# Asenkron dongu, JSON, zaman, os, torch ve web sunucusu kutuphaneleri yuklenir
import asyncio
import json
import os
import re
import subprocess
import time
from typing import List, Optional

import nest_asyncio
import torch
import uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from peft import PeftModel
from pydantic import BaseModel

# Jupyter icerisinde asenkron Uvicorn calistirabilmek icin nest_asyncio uygulanir
nest_asyncio.apply()

# Eger 'model' degiskeni bellekte yoksa (Colab yeni acildiysa) Drive'dan otomatik yukle
if "model" not in globals():
    print("[*] Model bellekte bulunamadi, Google Drive 'AI/TriageCore/lora_adapter' uzerinden yukleniyor...")
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
    MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR) if (ADAPTER_DIR / "tokenizer_config.json").exists() else MODEL_ID)
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
    if (ADAPTER_DIR / "adapter_config.json").exists():
        model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
        print("[+] Drive'daki egitilmis LoRA adaptoru basariyla yuklendi!")
    else:
        model = base_model
        print("[!] Drive'da adaptor bulunamadi, base model kullaniliyor.")

# Modeli degerlendirme (inference) moduna al
model.eval()

# Colab uzerinde calisacak FastAPI uygulamasi olusturulur
colab_app = FastAPI(title="TriageCore Remote Colab GPU Engine", version="2.0.0")
colab_app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

# Istek semalari tanimlanir
class ChatMessageItem(BaseModel):
    role: str
    content: str

class RemoteChatRequest(BaseModel):
    session_id: str = "default"
    mode: str = "triage"
    messages: List[ChatMessageItem]
    case_id: Optional[str] = None
    temperature: float = 0.2
    max_tokens: int = 512

def append_to_drive_log(entry: dict):
    """Her soruyu ve yaniti Google Drive 'AI/TriageCore/logs/colab_sessions.jsonl' dosyasina kaydeder."""
    LOGS_DIR.mkdir(parents=True, exist_ok=True)
    log_file = LOGS_DIR / "colab_sessions.jsonl"
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

@colab_app.get("/health")
def health():
    """Streamlit arayuzunun Colab GPU durumunu kontrol ettigi endpoint."""
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Colab-CPU"
    return {
        "status": "healthy",
        "model_loaded": True,
        "engine_type": "colab_gpu_qlora",
        "device": gpu_name,
        "version": "2.0.0",
    }

@colab_app.post("/v1/chat")
def generate_chat(req: RemoteChatRequest):
    """Cok turlu klinik triyaj ve Doktor-Hasta simulasyonu yanitlarini GPU'da uretir."""
    start_t = time.time()
    # Mesaj gecmisini modele uygun listeye cevir
    formatted_msgs = [{"role": m.role, "content": m.content} for m in req.messages]

    # Eger sistem istemi yoksa varsayilan Turkce klinik triyaj istemini en basa ekle
    has_system = any(m.role == "system" for m in req.messages)
    if not has_system:
        default_system = (
            "Sen TriageCore Acil Servis Klinik Karar Destek Asistanısın (CDSS). "
            "Görevin: Hastanın semptomlarını değerlendirmek, olası tanıları ve aksiyon planını DAİMA ve KESİNLİKLE TÜRKÇE olarak açıklamaktır. "
            "Asla İngilizce 'I am not a doctor' gibi kalıplar kullanma; doğrudan klinik uzman dille konuş. "
            "Yanıtını 'Klinik Değerlendirme', 'Önerilen Aksiyon Planı' ve 'ESI Aciliyet Seviyesi' formatında sun."
        )
        formatted_msgs.insert(0, {"role": "system", "content": default_system})

    # Llama-3 chat sablonunu uygula
    prompt = tokenizer.apply_chat_template(formatted_msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    # Gradyan hesaplamadan hizli uretim yap
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=req.max_tokens,
            temperature=max(0.05, req.temperature),
            do_sample=True,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
        )
    # Sadece yeni uretilen yanit tokenlarini coz (clean_up_tokenization_spaces uyarisi onlenir)
    reply_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
    elapsed_ms = round((time.time() - start_t) * 1000, 2)

    # Yanit metninden ESI aciliyet derecesini ve kirmizi bayraklari dinamik analiz et
    urgency_score = 3
    triage_level = "ESI-3: Urgent (Orta Derece Acil / 1 Saat Icinde)"
    is_emergency = False

    if any(k in reply_text for k in ["ESI Seviye 1", "ESI-1", "Resüsitasyon", "Kardiyak", "Miyokard"]):
        urgency_score = 1
        triage_level = "ESI-1: Resuscitation (Hayati Tehlike / Derhal Mudahale)"
        is_emergency = True
    elif any(k in reply_text for k in ["ESI Seviye 2", "ESI-2", "Acil Durum", "Apandisit", "Emboli"]):
        urgency_score = 2
        triage_level = "ESI-2: Emergency (Acil Servis / 10-15 Dk)"
        is_emergency = True
    elif any(k in reply_text for k in ["ESI Seviye 4", "ESI-4", "Yarı Acil", "Poliklinik", "Sistit", "Öksürük", "Grip"]):
        urgency_score = 4
        triage_level = "ESI-4: Less Urgent (Yarı Acil / Poliklinik Başvurusu)"
        is_emergency = False
    elif any(k in reply_text for k in ["ESI Seviye 5", "ESI-5", "Rutin"]):
        urgency_score = 5
        triage_level = "ESI-5: Non-Urgent (Acil Değil / Rutin Takip)"
        is_emergency = False

    # Yaniti Google Drive loglarina kaydet
    log_record = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "session_id": req.session_id,
        "mode": req.mode,
        "user_message": req.messages[-1].content if req.messages else "",
        "assistant_reply": reply_text,
        "triage_level": triage_level,
        "urgency_score": urgency_score,
        "inference_time_ms": elapsed_ms,
    }
    append_to_drive_log(log_record)

    return {
        "session_id": req.session_id,
        "mode": req.mode,
        "reply": reply_text,
        "triage_level": triage_level,
        "urgency_score": urgency_score,
        "is_emergency": is_emergency,
        "red_flags": [],
        "inference_time_ms": elapsed_ms,
    }

# ==============================================================================
# TUNEL BASLATICI: Google Colab Gizli Anahtarlar (Secrets) veya Cloudflared
# ==============================================================================
# Token baslangicta bos olarak tanimlanir
NGROK_AUTH_TOKEN = ""

# 1. Oncelikle Google Colab Secrets (Sol paneldeki anahtar simgesi) kontrol edilir
try:
    # Google Colab userdata modulu ice aktarilir
    from google.colab import userdata
    # 'NGROK_AUTH_TOKEN' anahtari Colab kasasindan guvenle okunur
    NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
    if NGROK_AUTH_TOKEN:
        print("[+] 'NGROK_AUTH_TOKEN' Google Colab Gizli Anahtarlarindan (Secrets) basariyla alindi.")
except Exception:
    # Colab disinda calisiliyorsa veya Secrets uzerinde anahtara izin verilmediyse
    pass

# 2. Eger Colab Secrets'tan alinamadiysa sistem ortam degiskenleri kontrol edilir
if not NGROK_AUTH_TOKEN:
    NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTH_TOKEN", "")
    if NGROK_AUTH_TOKEN:
        print("[+] 'NGROK_AUTH_TOKEN' sistem ortam degiskenlerinden (os.getenv) basariyla alindi.")

# 3. Token varsa Ngrok tuneli acilir; yoksa ucretsiz Cloudflared tuneline gecilir
if NGROK_AUTH_TOKEN:
    # pyngrok modulu ice aktarilir
    from pyngrok import ngrok
    # Ngrok yetkilendirme anahtari tanimlanir
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    # FastAPI portuna (8000) bagli genel HTTPS tuneli baslatilir
    public_url = ngrok.connect(8000).public_url
    print("\n" + "=" * 75)
    print(f"  [NGROK AKTIF] BAGLANTI ADRESINIZ: {public_url}")
    print("  Bu URL'yi kopyalayip yerel Streamlit arayuzunuzdeki Sol Panele yapistirin!")
    print("=" * 75 + "\n")
else:
    print("[*] Ngrok token bulunamadi, sifir ayar gerektiren Cloudflared HTTPS tuneli aciliyor...")
    # Cloudflared ikili dosyasi indirilir ve calistirma yetkisi verilir
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared-linux-amd64
    # Arka planda Cloudflared tuneli calistirilir
    tunnel_proc = subprocess.Popen(
        ["./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    # Standart ciktidan genel Cloudflare URL'si yakalanir
    for line in tunnel_proc.stdout:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            print("\n" + "=" * 75)
            print(f"  [CLOUDFLARE TUNNEL AKTIF] BAGLANTI ADRESINIZ: {public_url}")
            print("  Bu URL'yi kopyalayip yerel Streamlit arayuzunuzdeki Sol Panele yapistirin!")
            print("=" * 75 + "\n")
            break

# ==============================================================================
# FASTAPI / UVICORN SUNUCUSUNU CALISTIRMA (Python 3.13 & Jupyter Event Loop Uyumlu)
# ==============================================================================
# Google Colab ve Jupyter icerisinde zaten aktif bir asyncio event loop calistigi icin
# uvicorn.run() cagrildiginda 'asyncio.run() cannot be called from a running event loop'
# hatasi firlatir. Bu yuzden sunucu uvicorn.Server nesnesi olarak olusturulup dogrudan
# Jupyter kernel event loop'u uzerinde await ile kesintisiz calistirilir.
config = uvicorn.Config(app=colab_app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)

print("[+] FastAPI sunucusu 8000 portunda basariyla dinlemeye basladi!")
print("[*] Sunucu kesintisiz hizmet veriyor. Oturumu durdurmak icin sol ustteki Durdur (Stop) dugmesine basabilirsiniz.\n")

try:
    await server.serve()
except (KeyboardInterrupt, asyncio.CancelledError):
    print("\n[*] TriageCore AI sunucusu kullanici tarafindan durduruldu.")
